In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

df = pd.read_csv("data/tickets.csv")
df = df.fillna("")
df["text"] = df["product"] + " | " + df["channel"] + " | " + df["subject"] + " | " + df["body"]
df["text"].iloc[0]

'GreenLake | chat | console down for everyone | The GreenLake API is returning 429 on every call, our provisioning pipeline is completely stopped. It hosts our core banking VMs. HAR file attached. This is unacceptable. Please send an engineer.'

In [2]:
train, val = train_test_split(df, test_size=0.2, random_state=42)
print(len(train), len(val))

24000 6000


In [3]:
def make_model():
    features = make_union(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True),
    )
    return make_pipeline(features, LogisticRegression(max_iter=2000, C=5))

In [4]:
cat_model = make_model()
cat_model.fit(train["text"], train["category"])

pred = cat_model.predict(val["text"])
print("Category macro-F1:", round(f1_score(val["category"], pred, average="macro"), 3))
print(classification_report(val["category"], pred))

Category macro-F1: 0.865
              precision    recall  f1-score   support

      Access       0.85      0.83      0.84       718
     Billing       0.90      0.96      0.93       494
    Firmware       0.85      0.78      0.81       678
    Hardware       0.84      0.88      0.86      1001
  Networking       0.89      0.94      0.91       862
    Security       0.86      0.81      0.84       610
    Software       0.85      0.84      0.84       835
     Storage       0.88      0.87      0.88       802

    accuracy                           0.86      6000
   macro avg       0.87      0.86      0.86      6000
weighted avg       0.86      0.86      0.86      6000



In [8]:
prio_model = make_model()
prio_model.fit(train["text"], train["priority"])

pred = prio_model.predict(val["text"])
print("Priority macro-F1:", round(f1_score(val["priority"], pred, average="macro"), 3))
print(classification_report(val["priority"], pred))

Priority macro-F1: 0.759
              precision    recall  f1-score   support

          P1       0.78      0.65      0.71       708
          P2       0.70      0.74      0.72      1725
          P3       0.72      0.76      0.74      1869
          P4       0.90      0.85      0.87      1698

    accuracy                           0.77      6000
   macro avg       0.77      0.75      0.76      6000
weighted avg       0.77      0.77      0.77      6000



In [9]:
team_model = make_model()
team_model.fit(train["text"], train["assigned_team"])

pred = team_model.predict(val["text"])
print("Assigned team macro-F1:", round(f1_score(val["assigned_team"], pred, average="macro"), 3))
print(classification_report(val["assigned_team"], pred))

Assigned team macro-F1: 0.906
                  precision    recall  f1-score   support

   Account-Admin       0.86      0.87      0.87       786
     Billing-Ops       0.93      0.96      0.95       506
 Network-Support       0.92      0.92      0.92      1016
    Security-Ops       0.89      0.81      0.85       535
       Server-HW       0.92      0.95      0.94      1524
Software-Support       0.87      0.86      0.86       819
 Storage-Support       0.95      0.96      0.96       814

        accuracy                           0.91      6000
       macro avg       0.91      0.90      0.91      6000
    weighted avg       0.91      0.91      0.91      6000



In [10]:
senti_model = make_model()
senti_model.fit(train["text"], train["sentiment"])

pred = senti_model.predict(val["text"])
print("Sentiment macro-F1:", round(f1_score(val["sentiment"], pred, average="macro"), 3))
print(classification_report(val["sentiment"], pred))

Sentiment macro-F1: 0.869
              precision    recall  f1-score   support

       Angry       0.97      0.86      0.91      1004
  Frustrated       0.89      0.79      0.84      1471
     Neutral       0.79      0.95      0.86      2450
    Positive       0.98      0.78      0.87      1075

    accuracy                           0.86      6000
   macro avg       0.91      0.84      0.87      6000
weighted avg       0.88      0.86      0.86      6000



In [11]:
import numpy as np

misroute_cost = {"P1": 8, "P2": 2, "P3": 1, "P4": 1}
abstain_cost  = {"P1": 1.0, "P2": 0.5, "P3": 0.3, "P4": 0.3}

def routing_cost(true_team, true_prio, pred_team, abstain):
    total = 0
    for t, p, pt, a in zip(true_team, true_prio, pred_team, abstain):
        if a:
            total += abstain_cost[p]
        elif t != pt:
            total += misroute_cost[p]
    return total / len(true_team)

In [12]:
team_probs = team_model.predict_proba(val["text"])
prio_probs = prio_model.predict_proba(val["text"])
pred_team = team_model.classes_[team_probs.argmax(axis=1)]

no_abstain = np.zeros(len(val), dtype=bool)
print("Cost, never abstain:", round(routing_cost(val["assigned_team"], val["priority"], pred_team, no_abstain), 4))

Cost, never abstain: 0.1738


In [13]:
mis_vec = np.array([misroute_cost[p] for p in prio_model.classes_])
abs_vec = np.array([abstain_cost[p] for p in prio_model.classes_])

p_team_right = team_probs.max(axis=1)
cost_if_route   = (1 - p_team_right) * (prio_probs @ mis_vec)
cost_if_abstain = prio_probs @ abs_vec
saving = cost_if_route - cost_if_abstain

# very short tickets ("it broke") -> always send to a human
n_words = val["body"].str.split().str.len().values
saving[n_words < 5] = 99

limit = int(0.15 * len(val))
order = np.argsort(-saving)
abstain = np.zeros(len(val), dtype=bool)
abstain[order[:limit]] = saving[order[:limit]] > 0

print("Cost, smart abstain:", round(routing_cost(val["assigned_team"], val["priority"], pred_team, abstain), 4))
print("Abstain rate:", round(abstain.mean(), 3))

Cost, smart abstain: 0.1484
Abstain rate: 0.15


In [27]:
def triage(tickets, use_cap=True):
    t = tickets.fillna("").copy()
    text = t["product"] + " | " + t["channel"] + " | " + t["subject"] + " | " + t["body"]

    cat  = cat_model.predict(text)
    sent = senti_model.predict(text)
    prio_p = prio_model.predict_proba(text)
    prio   = prio_model.classes_[prio_p.argmax(axis=1)]
    team_p = team_model.predict_proba(text)
    team   = team_model.classes_[team_p.argmax(axis=1)]
    p_right = team_p.max(axis=1)

    # NEW - Fix 1: GreenLake SSO/MFA/API-key/role issues go to Account-Admin (README rule)
    low = (t["subject"] + " " + t["body"]).str.lower()
    acct = (t["product"] == "GreenLake").values & \
           low.str.contains(r"sso|saml|mfa|okta|api key|api-key|role|permission").values
    cat     = np.where(acct, "Access", cat)
    team    = np.where(acct, "Account-Admin", team)
    p_right = np.where(acct, 0.95, p_right)

    saving = (1 - p_right) * (prio_p @ mis_vec) - prio_p @ abs_vec

    # NEW - Fix 2: vague tickets under 10 words always go to a human
    n_words = t["body"].str.split().str.len().values
    saving[n_words < 10] = 99

    if use_cap:
        limit = int(0.145 * len(t))
        order = np.argsort(-saving)
        abstain = np.zeros(len(t), dtype=bool)
        abstain[order[:limit]] = saving[order[:limit]] > 0
    else:
        abstain = saving > 0

    out = pd.DataFrame({
        "ticket_id": t["ticket_id"], "category": cat, "priority": prio,
        "assigned_team": team, "sentiment": sent, "abstain": abstain.astype(int),
    })
    out.loc[out["abstain"] == 1, ["category", "assigned_team"]] = ""
    return out

In [29]:
ev = pd.read_csv("eval/eval_tickets.csv")
sub = triage(ev, use_cap=True)
sub.to_csv("submission.csv", index=False)

print("Rows:", len(sub), "| Expected:", len(ev))
print("Abstain rate:", round(sub["abstain"].mean(), 3))
sub.head()

Rows: 6000 | Expected: 6000
Abstain rate: 0.145


,ticket_id,category,priority,assigned_team,sentiment,abstain
0,TK-500001,Networking,P3,Network-Support,Positive,0
1,TK-500002,Networking,P3,Network-Support,Frustrated,0
2,TK-500003,Networking,P3,Network-Support,Positive,0
3,TK-500004,Storage,P1,Storage-Support,Frustrated,0
4,TK-500005,Storage,P1,Storage-Support,Positive,0


In [31]:
cat_model = make_model()
cat_model.fit(train["text"], train["category"])

pred = cat_model.predict(val["text"])
print("Category macro-F1:", round(f1_score(val["category"], pred, average="macro"), 3))
print(cat_model.classes_)

Category macro-F1: 0.865
['Access' 'Billing' 'Firmware' 'Hardware' 'Networking' 'Security'
 'Software' 'Storage']


In [33]:
cb = pd.read_csv("eval/curveball_tickets.csv")
sub_cb = triage(cb, use_cap=False)
sub_cb.to_csv("submission_curveball.csv", index=False)

view = sub_cb.copy()
view["subject"] = cb["subject"]
print(view[["ticket_id", "subject", "category", "priority", "assigned_team", "sentiment", "abstain"]].to_string())

   ticket_id                         subject    category priority    assigned_team sentiment  abstain
0     CB-001               Thank you SO much    Hardware       P2        Server-HW     Angry        0
1     CB-002                        it broke                   P4                    Neutral        1
2     CB-003           server tak boleh boot    Hardware       P2        Server-HW   Neutral        0
3     CB-004         server room aircon dead                   P2                    Neutral        1
4     CB-005                volume read-only     Storage       P1  Storage-Support   Neutral        0
5     CB-006                       URGENT!!!    Hardware       P4        Server-HW   Neutral        0
6     CB-007           AOS-CX upgrade failed    Firmware       P2  Network-Support   Neutral        0
7     CB-008  SSO broken after cert rotation      Access       P1    Account-Admin     Angry        0
8     CB-009         great news + a question     Billing       P4      Billing-Ops

In [25]:
low = (df["subject"] + " " + df["body"]).str.lower()
m = (df["product"] == "GreenLake") & low.str.contains(r"sso|saml|mfa|okta|api key|api-key|role|permission")
print(m.sum(), "matching tickets")
df[m]["assigned_team"].value_counts()

952 matching tickets


assigned_team
Account-Admin       847
Security-Ops         55
Software-Support     21
Billing-Ops          19
Network-Support      10
Name: count, dtype: int64

In [26]:
ev_words = ev["body"].str.split().str.len()
for n in [5, 8, 10]:
    print(f"under {n} words:", (ev_words < n).sum())
ev[ev_words < 10][["subject", "body"]].head(15)

under 5 words: 101
under 8 words: 101
under 10 words: 104


,subject,body
64,help,same problem
70,problem,it broke
79,?,same as before
137,urgent,it's doing it again
192,broken,broken since this morning
210,help,fix pls
225,help,still broken
286,?,pls help
390,Aruba CX 6300,same problem
562,broken,broken since this morning


In [28]:
v = triage(val, use_cap=True)
print("Validation cost with rules:", round(routing_cost(val["assigned_team"], val["priority"], v["assigned_team"], v["abstain"]), 4))
print("Abstain rate:", round(v["abstain"].mean(), 3))

Validation cost with rules: 0.1488
Abstain rate: 0.145


In [34]:
import joblib
joblib.dump(
    {"cat": cat_model, "prio": prio_model, "team": team_model, "senti": senti_model},
    "models.joblib",
    compress=3,
)
print("Saved!")

Saved!


In [35]:
import joblib
m = joblib.load("models.joblib")
print(m.keys())
print(m["cat"].classes_)

dict_keys(['cat', 'prio', 'team', 'senti'])
['Access' 'Billing' 'Firmware' 'Hardware' 'Networking' 'Security'
 'Software' 'Storage']
